# Model Training: Airline Passenger Satisfaction

**Dataset**: Airline Passenger Satisfaction  
**Objective**: Train, evaluate, and compare multiple ML classification models  
**Output**: JSON reports saved to `reports/model_training/`, model PKL files saved to `model/resources/`

In [12]:
import pandas as pd
import numpy as np
import json
import os
import sys
import pickle
from datetime import datetime

# Set working directory to project root
while not os.path.exists('requirements.txt') and os.getcwd() != os.path.dirname(os.getcwd()):
    os.chdir('..')

# Add model/ to path so metric_calculation can be imported
sys.path.insert(0, os.path.join(os.getcwd(), 'model'))
import metric_calculation as mc

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import confusion_matrix

# Set up reports directory
reports_dir = "reports/model_training"
os.makedirs(reports_dir, exist_ok=True)
os.makedirs("model/resources", exist_ok=True)

print(f"Working directory: {os.getcwd()}")
print(f"Model training report directory: {reports_dir}")
print(f"All JSON reports will be saved to: {reports_dir}/")

Working directory: c:\Users\finle\Git_projects\ML_Assingnment_2
Model training report directory: reports/model_training
All JSON reports will be saved to: reports/model_training/


## 1. Load Preprocessed Data

> **Note**: Run `data_preprocessing.ipynb` first to generate the preprocessed pickle files.

In [13]:
# Load preprocessed data from data_preprocessing.ipynb output
preprocessed_dir = "dataset/preprocessed"

with open(f"{preprocessed_dir}/X_train_scaled.pkl", "rb") as f:
    X_train_scaled = pickle.load(f)

with open(f"{preprocessed_dir}/X_test_scaled.pkl", "rb") as f:
    X_test_scaled = pickle.load(f)

with open(f"{preprocessed_dir}/X_train.pkl", "rb") as f:
    X_train = pickle.load(f)

with open(f"{preprocessed_dir}/X_test.pkl", "rb") as f:
    X_test = pickle.load(f)

with open(f"{preprocessed_dir}/y_train.pkl", "rb") as f:
    y_train = pickle.load(f)

with open(f"{preprocessed_dir}/y_test.pkl", "rb") as f:
    y_test = pickle.load(f)

with open("model/resources/scaler.pkl", "rb") as f:
    scaler = pickle.load(f)

with open("model/resources/feature_columns.pkl", "rb") as f:
    feature_columns = pickle.load(f)

print("Preprocessed data loaded successfully!")
print(f"X_train_scaled: {X_train_scaled.shape}")
print(f"X_test_scaled:  {X_test_scaled.shape}")
print(f"X_train:        {X_train.shape}")
print(f"X_test:         {X_test.shape}")
print(f"y_train:        {y_train.shape}")
print(f"y_test:         {y_test.shape}")
print(f"Features:       {len(feature_columns)}")
print(f"Train satisfaction ratio: {y_train.mean():.4f}")
print(f"Test satisfaction ratio:  {y_test.mean():.4f}")

Preprocessed data loaded successfully!
X_train_scaled: (79123, 23)
X_test_scaled:  (19781, 23)
X_train:        (79123, 23)
X_test:         (19781, 23)
y_train:        (79123,)
y_test:         (19781,)
Features:       23
Train satisfaction ratio: 0.4330
Test satisfaction ratio:  0.4329


In [14]:
# Save data preparation summary as JSON
data_preparation = {
    "timestamp": datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    "data_source": "Loaded from data_preprocessing.ipynb output (dataset/preprocessed/)",
    "original_dataset": "dataset/test_original.csv",
    "preprocessing_notebook": "data_preprocessing.ipynb",
    "X_train_shape": list(X_train_scaled.shape),
    "X_test_shape": list(X_test_scaled.shape),
    "train_satisfaction_ratio": round(float(y_train.mean()), 4),
    "test_satisfaction_ratio": round(float(y_test.mean()), 4),
    "feature_columns": feature_columns.tolist(),
    "total_features": len(feature_columns)
}

with open(f"{reports_dir}/data_preparation.json", "w") as f:
    json.dump(data_preparation, f, indent=2)

print(f"Saved: {reports_dir}/data_preparation.json")

Saved: reports/model_training/data_preparation.json


## 3. Train and Evaluate Models

In [15]:
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_scaled, y_train)

# Make predictions
y_pred_lr = lr.predict(X_test_scaled)
y_pred_proba_lr = lr.predict_proba(X_test_scaled)[:, 1]

# Calculate and display metrics using metric_calculation module
lr_metrics = mc.calculate_metrics(y_test, y_pred_lr, y_pred_proba_lr)
mc.display_metrics("Logistic Regression", lr_metrics)

# Confusion matrix
lr_cm = confusion_matrix(y_test, y_pred_lr).tolist()

# Save metrics to JSON
lr_report = {
    "model_name": "Logistic Regression",
    "timestamp": datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    "hyperparameters": {"max_iter": 1000},
    "uses_scaled_data": True,
    "metrics": lr_metrics,
    "confusion_matrix": lr_cm
}

with open(f"{reports_dir}/logistic_regression.json", "w") as f:
    json.dump(lr_report, f, indent=2)

print(f"\nSaved: {reports_dir}/logistic_regression.json")


Model: Logistic Regression
Accuracy                  : 0.8781
AUC Score                 : 0.9296
Precision                 : 0.8730
Recall                    : 0.8406
F1 Score                  : 0.8565
MCC Score                 : 0.7510


Saved: reports/model_training/logistic_regression.json


In [16]:
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)

# Make predictions
y_pred_dt = dt.predict(X_test)
y_pred_proba_dt = dt.predict_proba(X_test)[:, 1]

# Calculate and display metrics using metric_calculation module
dt_metrics = mc.calculate_metrics(y_test, y_pred_dt, y_pred_proba_dt)
mc.display_metrics("Decision Tree", dt_metrics)

# Confusion matrix
dt_cm = confusion_matrix(y_test, y_pred_dt).tolist()

# Save metrics to JSON
dt_report = {
    "model_name": "Decision Tree",
    "timestamp": datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    "hyperparameters": {"random_state": 42},
    "uses_scaled_data": False,
    "metrics": dt_metrics,
    "confusion_matrix": dt_cm
}

with open(f"{reports_dir}/decision_tree.json", "w") as f:
    json.dump(dt_report, f, indent=2)

print(f"\nSaved: {reports_dir}/decision_tree.json")


Model: Decision Tree
Accuracy                  : 0.9449
AUC Score                 : 0.9439
Precision                 : 0.9366
Recall                    : 0.9361
F1 Score                  : 0.9363
MCC Score                 : 0.8878


Saved: reports/model_training/decision_tree.json


In [17]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)

# Make predictions
y_pred_knn = knn.predict(X_test_scaled)
y_pred_proba_knn = knn.predict_proba(X_test_scaled)[:, 1]

# Calculate and display metrics using metric_calculation module
knn_metrics = mc.calculate_metrics(y_test, y_pred_knn, y_pred_proba_knn)
mc.display_metrics("K-Nearest Neighbors", knn_metrics)

# Confusion matrix
knn_cm = confusion_matrix(y_test, y_pred_knn).tolist()

# Save metrics to JSON
knn_report = {
    "model_name": "K-Nearest Neighbors",
    "timestamp": datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    "hyperparameters": {"n_neighbors": 5},
    "uses_scaled_data": True,
    "metrics": knn_metrics,
    "confusion_matrix": knn_cm
}

with open(f"{reports_dir}/knn.json", "w") as f:
    json.dump(knn_report, f, indent=2)

print(f"\nSaved: {reports_dir}/knn.json")


Model: K-Nearest Neighbors
Accuracy                  : 0.9272
AUC Score                 : 0.9685
Precision                 : 0.9501
Recall                    : 0.8779
F1 Score                  : 0.9125
MCC Score                 : 0.8522


Saved: reports/model_training/knn.json


In [18]:
nb = GaussianNB()
nb.fit(X_train_scaled, y_train)

# Make predictions
y_pred_nb = nb.predict(X_test_scaled)
y_pred_proba_nb = nb.predict_proba(X_test_scaled)[:, 1]

# Calculate and display metrics using metric_calculation module
nb_metrics = mc.calculate_metrics(y_test, y_pred_nb, y_pred_proba_nb)
mc.display_metrics("Naive Bayes", nb_metrics)

# Confusion matrix
nb_cm = confusion_matrix(y_test, y_pred_nb).tolist()

# Save metrics to JSON
nb_report = {
    "model_name": "Naive Bayes",
    "timestamp": datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    "hyperparameters": {},
    "uses_scaled_data": True,
    "metrics": nb_metrics,
    "confusion_matrix": nb_cm
}

with open(f"{reports_dir}/naive_bayes.json", "w") as f:
    json.dump(nb_report, f, indent=2)

print(f"\nSaved: {reports_dir}/naive_bayes.json")


Model: Naive Bayes
Accuracy                  : 0.8625
AUC Score                 : 0.9214
Precision                 : 0.8584
Recall                    : 0.8174
F1 Score                  : 0.8374
MCC Score                 : 0.7191


Saved: reports/model_training/naive_bayes.json


In [19]:
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)
rf.fit(X_train, y_train)

# Make predictions
y_pred_rf = rf.predict(X_test)
y_pred_proba_rf = rf.predict_proba(X_test)[:, 1]

# Calculate and display metrics using metric_calculation module
rf_metrics = mc.calculate_metrics(y_test, y_pred_rf, y_pred_proba_rf)
mc.display_metrics("Random Forest", rf_metrics)

# Confusion matrix
rf_cm = confusion_matrix(y_test, y_pred_rf).tolist()

# Save metrics to JSON
rf_report = {
    "model_name": "Random Forest",
    "timestamp": datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    "hyperparameters": {"n_estimators": 100, "random_state": 42},
    "uses_scaled_data": False,
    "metrics": rf_metrics,
    "confusion_matrix": rf_cm
}

with open(f"{reports_dir}/random_forest.json", "w") as f:
    json.dump(rf_report, f, indent=2)

print(f"\nSaved: {reports_dir}/random_forest.json")


Model: Random Forest
Accuracy                  : 0.9617
AUC Score                 : 0.9936
Precision                 : 0.9730
Recall                    : 0.9376
F1 Score                  : 0.9550
MCC Score                 : 0.9222


Saved: reports/model_training/random_forest.json


In [20]:
xgb_model = XGBClassifier(
    eval_metric='logloss',
    random_state=42
)
xgb_model.fit(X_train, y_train)

# Make predictions
y_pred_xgb = xgb_model.predict(X_test)
y_pred_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]

# Calculate and display metrics using metric_calculation module
xgb_metrics = mc.calculate_metrics(y_test, y_pred_xgb, y_pred_proba_xgb)
mc.display_metrics("XGBoost", xgb_metrics)

# Confusion matrix
xgb_cm = confusion_matrix(y_test, y_pred_xgb).tolist()

# Save metrics to JSON
xgb_report = {
    "model_name": "XGBoost",
    "timestamp": datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    "hyperparameters": {"eval_metric": "logloss", "random_state": 42},
    "uses_scaled_data": False,
    "metrics": xgb_metrics,
    "confusion_matrix": xgb_cm
}

with open(f"{reports_dir}/xgboost.json", "w") as f:
    json.dump(xgb_report, f, indent=2)

print(f"\nSaved: {reports_dir}/xgboost.json")


Model: XGBoost
Accuracy                  : 0.9619
AUC Score                 : 0.9948
Precision                 : 0.9704
Recall                    : 0.9407
F1 Score                  : 0.9553
MCC Score                 : 0.9224


Saved: reports/model_training/xgboost.json


## 4. Model Comparison

In [21]:
results_df = pd.DataFrame.from_dict({
    "Logistic Regression": lr_metrics,
    "Decision Tree": dt_metrics,
    "KNN": knn_metrics,
    "Naive Bayes": nb_metrics,
    "Random Forest": rf_metrics,
    "XGBoost": xgb_metrics
}, orient='index')

print("\n" + "="*60)
print("MODEL COMPARISON (sorted by F1 Score)")
print("="*60)
results_sorted = results_df.sort_values(by="F1 Score", ascending=False)
print(results_sorted)

# Save model comparison to JSON
model_comparison = {
    "timestamp": datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    "models_compared": list(results_df.index),
    "ranking_metric": "F1 Score",
    "results": {
        model: {metric: round(float(val), 4) for metric, val in row.items()}
        for model, row in results_df.iterrows()
    },
    "best_model": results_sorted.index[0],
    "best_f1_score": round(float(results_sorted["F1 Score"].iloc[0]), 4)
}

with open(f"{reports_dir}/model_comparison.json", "w") as f:
    json.dump(model_comparison, f, indent=2)

print(f"\nSaved: {reports_dir}/model_comparison.json")
results_sorted


MODEL COMPARISON (sorted by F1 Score)
                     Accuracy  AUC Score  Precision    Recall  F1 Score  \
XGBoost              0.961883   0.994799   0.970369  0.940682  0.955295   
Random Forest        0.961731   0.993640   0.972980  0.937646  0.954986   
Decision Tree        0.944897   0.943860   0.936565  0.936128  0.936347   
KNN                  0.927152   0.968503   0.950082  0.877861  0.912545   
Logistic Regression  0.878065   0.929607   0.873029  0.840612  0.856514   
Naive Bayes          0.862545   0.921357   0.858369  0.817375  0.837371   

                     MCC Score  
XGBoost               0.922425  
Random Forest         0.922201  
Decision Tree         0.887768  
KNN                   0.852242  
Logistic Regression   0.750973  
Naive Bayes           0.719109  

Saved: reports/model_training/model_comparison.json


,Accuracy,AUC Score,Precision,Recall,F1 Score,MCC Score
XGBoost,0.961883,0.994799,0.970369,0.940682,0.955295,0.922425
Random Forest,0.961731,0.993640,0.972980,0.937646,0.954986,0.922201
Decision Tree,0.944897,0.943860,0.936565,0.936128,0.936347,0.887768
KNN,0.927152,0.968503,0.950082,0.877861,0.912545,0.852242
Logistic Regression,0.878065,0.929607,0.873029,0.840612,0.856514,0.750973
Naive Bayes,0.862545,0.921357,0.858369,0.817375,0.837371,0.719109


## 5. Save Models and Training Summary

In [22]:
os.makedirs("model/resources", exist_ok=True)

# Save scaler
with open("model/resources/scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

# Save models
models = {
    "logistic_regression.pkl": lr,
    "decision_tree.pkl": dt,
    "knn.pkl": knn,
    "naive_bayes.pkl": nb,
    "random_forest.pkl": rf,
    "xgboost.pkl": xgb_model
}

for filename, model in models.items():
    with open(f"model/resources/{filename}", "wb") as f:
        pickle.dump(model, f)

print("All models and scaler saved successfully to model/resources/ directory.")

# Save training summary as JSON
json_files = [
    "data_preparation.json",
    "logistic_regression.json",
    "decision_tree.json",
    "knn.json",
    "naive_bayes.json",
    "random_forest.json",
    "xgboost.json",
    "model_comparison.json",
    "training_summary.json"
]

pkl_files = ["scaler.pkl", "feature_columns.pkl"] + list(models.keys())

training_summary = {
    "training_date": datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    "dataset": "dataset/test_original.csv",
    "models_trained": list(models.keys()),
    "total_models": len(models),
    "best_model": results_sorted.index[0],
    "best_f1_score": round(float(results_sorted["F1 Score"].iloc[0]), 4),
    "pkl_files_saved": pkl_files,
    "json_files_generated": json_files,
    "total_json_files": len(json_files),
    "total_pkl_files": len(pkl_files)
}

with open(f"{reports_dir}/training_summary.json", "w") as f:
    json.dump(training_summary, f, indent=2)

print(f"\nSaved: {reports_dir}/training_summary.json")
print(f"\nTotal JSON reports: {len(json_files)}")
print(f"Total PKL files: {len(pkl_files)}")
for jf in json_files:
    print(f"  - {reports_dir}/{jf}")
for pf in pkl_files:
    print(f"  - model/resources/{pf}")

All models and scaler saved successfully to model/resources/ directory.

Saved: reports/model_training/training_summary.json

Total JSON reports: 9
Total PKL files: 8
  - reports/model_training/data_preparation.json
  - reports/model_training/logistic_regression.json
  - reports/model_training/decision_tree.json
  - reports/model_training/knn.json
  - reports/model_training/naive_bayes.json
  - reports/model_training/random_forest.json
  - reports/model_training/xgboost.json
  - reports/model_training/model_comparison.json
  - reports/model_training/training_summary.json
  - model/resources/scaler.pkl
  - model/resources/feature_columns.pkl
  - model/resources/logistic_regression.pkl
  - model/resources/decision_tree.pkl
  - model/resources/knn.pkl
  - model/resources/naive_bayes.pkl
  - model/resources/random_forest.pkl
  - model/resources/xgboost.pkl
